# Phase 2 - Model Implementation and Training

This notebook completes model implementation, model training, hyperparameter tuning, model saving, and prediction export for hotspot forecasting.

## Model Implementation

Four scikit-learn classification models are implemented:

- Logistic Regression
- Random Forest
- Gradient Boosting
- HistGradientBoosting

These models provide a balanced comparison between a simple linear baseline and stronger non-linear ensemble methods. XGBoost was not used here because the local environment did not contain the package, while the four selected models can run directly in a fully reproducible way.

The prediction target is `target_hotspot_next_block`, which indicates whether a district becomes a hotspot in the next 4-hour block. Predictors include the district identifier, temporal indicators, current crime conditions, lagged crime counts, and rolling historical averages.

## Training Process

The notebook uses `train_panel_featured.csv` and `test_panel_featured.csv` generated in Phase 2. To avoid temporal leakage, a time-based validation split is used during tuning: the earliest 80% of training dates are used for model fitting and the latest 20% of training dates are used for validation. After the best hyperparameters are selected, each model is retrained on the full training set and then evaluated on the 2025 test set.

The final exported artifacts are:

- `models/*.joblib`: trained model files
- `predictions/*_test_predictions.csv`: predicted probabilities and labels
- `reports/model_comparison_metrics.csv`: final model comparison table
- `reports/hyperparameter_tuning_results.csv`: full tuning log
- `reports/best_params.json`: selected best parameter combinations

## Hyperparameter Tuning

A compact grid search is used for each model. The selection metric is validation F1-score, because hotspot prediction is an imbalanced binary classification task and F1-score provides a more balanced view than accuracy alone. Logistic Regression tunes regularization strength and class weighting. Random Forest tunes tree depth, leaf size, and class weighting. Gradient Boosting tunes the number of estimators, learning rate, and tree depth. HistGradientBoosting tunes the number of boosting iterations, learning rate, maximum depth, and minimum leaf size.

## AI Assistance Declaration

Generative AI was used in a limited supporting role in the preparation of this notebook.

Its use was limited to helping structure the workflow, refine methodological explanations, suggest code templates for preprocessing and feature engineering, and assist with interpretation of intermediate results.

All final decisions, execution of the notebook, validation of outputs, and integration into the project were completed by the student.

In [1]:
from itertools import product
from pathlib import Path
import json
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
TRAIN_PATH = PROJECT_ROOT / 'train_panel_featured.csv'
TEST_PATH = PROJECT_ROOT / 'test_panel_featured.csv'

MODELS_DIR = PROJECT_ROOT / 'models'
PREDICTIONS_DIR = PROJECT_ROOT / 'predictions'
REPORTS_DIR = PROJECT_ROOT / 'reports'

# 这一层是为了让 model_evaluation.ipynb 直接可读
DATA_DIR = PROJECT_ROOT / 'data'

TARGET_COL = 'target_hotspot_next_block'
DROP_COLS = {TARGET_COL, 'dataset', 'event_date'}

for path in (MODELS_DIR, PREDICTIONS_DIR, REPORTS_DIR, DATA_DIR):
    path.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN_PATH, parse_dates=['event_date'])
test_df = pd.read_csv(TEST_PATH, parse_dates=['event_date'])

train_df.shape, test_df.shape

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\AAA_nus\\Sem2\\IT5006\\project\\notebooks\\phase2\\train_panel_featured.csv'

In [6]:
feature_cols = [col for col in train_df.columns if col not in DROP_COLS]
categorical_cols = ['district', 'time_block', 'day_of_week', 'month', 'quarter', 'is_weekend']

combined = pd.concat(
    [
        train_df[feature_cols].assign(_source='train'),
        test_df[feature_cols].assign(_source='test'),
    ],
    ignore_index=True,
)

encoded = pd.get_dummies(
    combined,
    columns=[col for col in categorical_cols if col in combined.columns],
    drop_first=False,
    dtype=float,
)

train_x = encoded.loc[encoded['_source'] == 'train'].drop(columns='_source').reset_index(drop=True)
test_x = encoded.loc[encoded['_source'] == 'test'].drop(columns='_source').reset_index(drop=True)
train_y = train_df[TARGET_COL].astype(int)
test_y = test_df[TARGET_COL].astype(int)

N_TIME_FOLDS = 3
INITIAL_TRAIN_RATIO = 0.5

unique_dates = np.array(sorted(train_df['event_date'].dt.normalize().unique()))
initial_train_size = max(1, int(len(unique_dates) * INITIAL_TRAIN_RATIO))
remaining_dates = len(unique_dates) - initial_train_size
fold_size = max(1, remaining_dates // N_TIME_FOLDS)
time_folds = []

for fold_idx in range(N_TIME_FOLDS):
    valid_start_idx = initial_train_size + fold_idx * fold_size
    if valid_start_idx >= len(unique_dates):
        break
    if fold_idx == N_TIME_FOLDS - 1:
        valid_end_idx = len(unique_dates)
    else:
        valid_end_idx = min(len(unique_dates), valid_start_idx + fold_size)

    valid_start_date = pd.Timestamp(unique_dates[valid_start_idx])
    valid_end_date = pd.Timestamp(unique_dates[valid_end_idx - 1])
    train_mask = (train_df['event_date'] < valid_start_date).to_numpy()
    valid_mask = ((train_df['event_date'] >= valid_start_date) & (train_df['event_date'] <= valid_end_date)).to_numpy()

    if train_mask.sum() == 0 or valid_mask.sum() == 0:
        continue

    time_folds.append({
        'fold': fold_idx + 1,
        'train_mask': train_mask,
        'valid_mask': valid_mask,
        'train_end_date': str((valid_start_date - pd.Timedelta(days=1)).date()),
        'valid_start_date': str(valid_start_date.date()),
        'valid_end_date': str(valid_end_date.date()),
        'train_rows': int(train_mask.sum()),
        'valid_rows': int(valid_mask.sum()),
    })

fold_summary = pd.DataFrame([
    {
        'fold': fold['fold'],
        'train_end_date': fold['train_end_date'],
        'valid_start_date': fold['valid_start_date'],
        'valid_end_date': fold['valid_end_date'],
        'train_rows': fold['train_rows'],
        'valid_rows': fold['valid_rows'],
    }
    for fold in time_folds
])

print('Feature count after encoding:', train_x.shape[1])
print('Number of temporal folds:', len(time_folds))
fold_summary


Feature count after encoding: 62
Number of temporal folds: 3


,fold,train_end_date,valid_start_date,valid_end_date,train_rows,valid_rows
0,1,2019-12-31,2020-01-01,2021-08-31,251988,84042
1,2,2021-08-31,2021-09-01,2023-05-02,336030,84042
2,3,2023-05-02,2023-05-03,2024-12-31,420072,84019


In [ ]:
def iter_param_grid(grid):
    keys = list(grid.keys())
    for values in product(*(grid[key] for key in keys)):
        yield dict(zip(keys, values))


def compute_metrics(y_true, probas, threshold=0.5):
    preds = (probas >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_true, preds),
        'precision': precision_score(y_true, preds, zero_division=0),
        'recall': recall_score(y_true, preds, zero_division=0),
        'f1': f1_score(y_true, preds, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probas),
    }


def find_best_threshold(y_true, probas, thresholds=None):
    """
    在验证集上扫描 threshold，返回 F1 最优阈值及其对应指标。
    若 F1 并列，则优先 recall 更高；再并列则选更接近 0.5 的阈值。
    """
    if thresholds is None:
        thresholds = np.arange(0.10, 0.91, 0.05)

    best_threshold = 0.5
    best_metrics = None

    for thr in thresholds:
        metrics = compute_metrics(y_true, probas, threshold=thr)

        if best_metrics is None:
            best_threshold = thr
            best_metrics = metrics
            continue

        better = False

        if metrics['f1'] > best_metrics['f1']:
            better = True
        elif np.isclose(metrics['f1'], best_metrics['f1']):
            if metrics['recall'] > best_metrics['recall']:
                better = True
            elif np.isclose(metrics['recall'], best_metrics['recall']):
                if abs(thr - 0.5) < abs(best_threshold - 0.5):
                    better = True

        if better:
            best_threshold = thr
            best_metrics = metrics

    return float(best_threshold), best_metrics


# 使用你 v2 的网格
model_specs = {
    'logistic_regression': {
        'builder': lambda params: LogisticRegression(
            max_iter=1200,
            solver='lbfgs',
            random_state=42,
            **params
        ),
        'grid': {
            'C': [0.01, 0.1, 1.0],
            'class_weight': [None, 'balanced'],
        },
        'scale': True,
    },
    'random_forest': {
        'builder': lambda params: RandomForestClassifier(
            random_state=42,
            n_jobs=1,
            **params
        ),
        'grid': {
            'n_estimators': [200],
            'max_depth': [12, 20],
            'min_samples_leaf': [1, 5],
            'class_weight': ['balanced_subsample'],
        },
        'scale': False,
    },
    'gradient_boosting': {
        'builder': lambda params: GradientBoostingClassifier(
            random_state=42,
            **params
        ),
        'grid': {
            'n_estimators': [150, 250],
            'learning_rate': [0.03, 0.05],
            'max_depth': [2],
            'subsample': [0.8],
        },
        'scale': False,
    },
    'hist_gradient_boosting': {
        'builder': lambda params: HistGradientBoostingClassifier(
            random_state=42,
            early_stopping=False,
            **params
        ),
        'grid': {
            'max_iter': [150, 250],
            'learning_rate': [0.03, 0.05],
            'max_depth': [8],
            'min_samples_leaf': [20, 50],
        },
        'scale': False,
    },
}

In [ ]:
def fit_and_export_all_models(train_x, test_x, train_y, test_y):
    tuning_rows = []
    fold_metric_rows = []
    metric_rows = []
    best_params = {}
    feature_names = list(train_x.columns)

    for model_name, spec in model_specs.items():
        print(f'Running {model_name}...')

        best_score = -1.0
        best_row = None

        for params in iter_param_grid(spec['grid']):
            started = time.time()
            fold_scores = []
            fold_thresholds = []

            for fold in time_folds:
                train_mask = fold['train_mask']
                valid_mask = fold['valid_mask']

                if spec['scale']:
                    scaler = StandardScaler()
                    scaler.fit(train_x.loc[train_mask])
                    x_train_part = scaler.transform(train_x.loc[train_mask]).astype(np.float32)
                    x_valid = scaler.transform(train_x.loc[valid_mask]).astype(np.float32)
                else:
                    x_train_part = train_x.loc[train_mask].to_numpy(dtype=np.float32)
                    x_valid = train_x.loc[valid_mask].to_numpy(dtype=np.float32)

                y_train_part = train_y.loc[train_mask]
                y_valid = train_y.loc[valid_mask]

                model = spec['builder'](params)
                model.fit(x_train_part, y_train_part)

                valid_proba = model.predict_proba(x_valid)[:, 1]

                best_threshold, fold_metrics = find_best_threshold(y_valid, valid_proba)

                fold_scores.append(fold_metrics)
                fold_thresholds.append(best_threshold)

                fold_metric_rows.append({
                    'model': model_name,
                    'fold': fold['fold'],
                    'train_end_date': fold['train_end_date'],
                    'valid_start_date': fold['valid_start_date'],
                    'valid_end_date': fold['valid_end_date'],
                    'train_rows': fold['train_rows'],
                    'valid_rows': fold['valid_rows'],
                    'best_threshold': best_threshold,
                    **params,
                    **{f'fold_{k}': v for k, v in fold_metrics.items()},
                })

            mean_metrics = {
                metric_name: float(np.mean([score[metric_name] for score in fold_scores]))
                for metric_name in fold_scores[0].keys()
            }
            mean_best_threshold = float(np.mean(fold_thresholds))

            row = {
                'model': model_name,
                'num_folds': len(fold_scores),
                'fit_seconds': round(time.time() - started, 2),
                'mean_best_threshold': mean_best_threshold,
                **params,
                **{f'valid_mean_{k}': v for k, v in mean_metrics.items()},
            }
            tuning_rows.append(row)

            if mean_metrics['f1'] > best_score:
                best_score = mean_metrics['f1']
                best_row = row

        best_params[model_name] = {
            'params': {key: best_row[key] for key in spec['grid'].keys()},
            'threshold': float(best_row['mean_best_threshold']),
        }

        final_scaler = None
        if spec['scale']:
            final_scaler = StandardScaler()
            final_scaler.fit(train_x)
            x_train_full = final_scaler.transform(train_x).astype(np.float32)
            x_test = final_scaler.transform(test_x).astype(np.float32)
        else:
            x_train_full = train_x.to_numpy(dtype=np.float32)
            x_test = test_x.to_numpy(dtype=np.float32)

        final_model = spec['builder'](best_params[model_name]['params'])

        started = time.time()
        final_model.fit(x_train_full, train_y)
        test_proba = final_model.predict_proba(x_test)[:, 1]

        final_threshold = best_params[model_name]['threshold']
        test_metrics = compute_metrics(test_y, test_proba, threshold=final_threshold)

        metric_rows.append({
            'model': model_name,
            'num_tuning_folds': len(time_folds),
            'full_train_rows': int(len(train_y)),
            'test_rows': int(len(test_y)),
            'fit_seconds': round(time.time() - started, 2),
            'decision_threshold': final_threshold,
            **best_params[model_name]['params'],
            **{f'test_{k}': v for k, v in test_metrics.items()},
        })

        artifact = {
            'model': final_model,
            'scaler': final_scaler,
            'feature_names': feature_names,
            'best_params': best_params[model_name]['params'],
            'decision_threshold': final_threshold,
        }
        joblib.dump(artifact, MODELS_DIR / f'{model_name}.joblib')

        prediction_df = test_df[['district', 'event_date', 'time_block', TARGET_COL]].copy()
        prediction_df['predicted_probability'] = test_proba
        prediction_df['predicted_label'] = (test_proba >= final_threshold).astype(int)
        prediction_df['decision_threshold'] = final_threshold
        prediction_df.rename(columns={TARGET_COL: 'actual_label'}, inplace=True)

        # 保留新代码原输出
        prediction_df.to_csv(PREDICTIONS_DIR / f'{model_name}_test_predictions.csv', index=False)

        # 额外导出给 model_evaluation.ipynb
        prediction_df.to_csv(DATA_DIR / f'test_predictions_{model_name}.csv', index=False)

    tuning_df = pd.DataFrame(tuning_rows).sort_values(
        ['model', 'valid_mean_f1', 'valid_mean_roc_auc'],
        ascending=[True, False, False]
    )

    fold_metrics_df = pd.DataFrame(fold_metric_rows).sort_values(['model', 'fold'])
    metrics_df = pd.DataFrame(metric_rows).sort_values(['test_f1', 'test_roc_auc'], ascending=[False, False])

    tuning_df.to_csv(REPORTS_DIR / 'hyperparameter_tuning_results.csv', index=False)
    fold_metrics_df.to_csv(REPORTS_DIR / 'time_fold_metrics.csv', index=False)
    metrics_df.to_csv(REPORTS_DIR / 'model_comparison_metrics.csv', index=False)
    (REPORTS_DIR / 'best_params.json').write_text(json.dumps(best_params, indent=2), encoding='utf-8')

    # 给评估 notebook 直接读取
    fold_metrics_df.to_csv(DATA_DIR / 'time_fold_metrics.csv', index=False)

    return tuning_df, metrics_df, best_params

In [8]:
tuning_results, metrics, best_params = fit_and_export_all_models(train_x, test_x, train_y, test_y)
metrics


Running logistic_regression...
Running random_forest...
Running gradient_boosting...
Running hist_gradient_boosting...


,model,num_tuning_folds,full_train_rows,test_rows,fit_seconds,C,class_weight,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,n_estimators,max_depth,min_samples_leaf,learning_rate,subsample,max_iter
1,random_forest,3,504091,50347,91.29,NaN,balanced_subsample,0.740402,0.389356,0.768994,0.516964,0.826985,100.0,12.0,5.0,NaN,NaN,NaN
0,logistic_regression,3,504091,50347,3.48,0.1,balanced,0.731444,0.379571,0.766905,0.507808,0.821740,NaN,NaN,NaN,NaN,NaN,NaN
3,hist_gradient_boosting,3,504091,50347,4.65,NaN,NaN,0.834091,0.570856,0.328642,0.417138,0.833516,NaN,8.0,50.0,0.1,NaN,120.0
2,gradient_boosting,3,504091,50347,115.29,NaN,NaN,0.831172,0.566689,0.277955,0.372971,0.821238,100.0,2.0,NaN,0.1,0.8,NaN


In [ ]:
print("=== data ===")
for p in sorted(DATA_DIR.glob("*")):
    print(p.name)

print("\n=== reports ===")
for p in sorted(REPORTS_DIR.glob("*")):
    print(p.name)

print("\n=== predictions ===")
for p in sorted(PREDICTIONS_DIR.glob("*")):
    print(p.name)

## 3. Post-Training Output Saving and Organization

### Why this section is needed
The core training and tuning step has already been completed in the previous cell. At this stage, the goal is not to retrain the models, but to organize the generated outputs into a clear handoff package for later submission, reporting, and model comparison.

### What this section does
- confirms that the trained model files have been saved
- confirms that the prediction result files have been saved
- reloads the exported summary files when needed
- presents a compact comparison table for reporting

### Output
- trained model artifacts in `models/`
- prediction files in `predictions/`
- metrics and tuning summaries in `reports/`


In [9]:
model_files = sorted(MODELS_DIR.glob('*.joblib'))
prediction_files = sorted(PREDICTIONS_DIR.glob('*_test_predictions.csv'))
report_files = sorted(REPORTS_DIR.glob('*'))

print('Saved model files:')
for path in model_files:
    print('-', path.name)

print('\nSaved prediction files:')
for path in prediction_files:
    print('-', path.name)

print('\nSaved report files:')
for path in report_files:
    print('-', path.name)


Saved model files:
- gradient_boosting.joblib
- hist_gradient_boosting.joblib
- logistic_regression.joblib
- random_forest.joblib

Saved prediction files:
- gradient_boosting_test_predictions.csv
- hist_gradient_boosting_test_predictions.csv
- logistic_regression_test_predictions.csv
- random_forest_test_predictions.csv

Saved report files:
- best_params.json
- hyperparameter_tuning_results.csv
- model_comparison_metrics.csv
- time_fold_metrics.csv


### 3.1 Review the Final Model Comparison Table

We now review the final comparison table exported after tuning and full training. This table is the main reference for discussing which model performed best on the 2025 test set.


In [10]:
metrics = pd.read_csv(REPORTS_DIR / 'model_comparison_metrics.csv')
metrics


,model,num_tuning_folds,full_train_rows,test_rows,fit_seconds,C,class_weight,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,n_estimators,max_depth,min_samples_leaf,learning_rate,subsample,max_iter
0,random_forest,3,504091,50347,91.29,NaN,balanced_subsample,0.740402,0.389356,0.768994,0.516964,0.826985,100.0,12.0,5.0,NaN,NaN,NaN
1,logistic_regression,3,504091,50347,3.48,0.1,balanced,0.731444,0.379571,0.766905,0.507808,0.821740,NaN,NaN,NaN,NaN,NaN,NaN
2,hist_gradient_boosting,3,504091,50347,4.65,NaN,NaN,0.834091,0.570856,0.328642,0.417138,0.833516,NaN,8.0,50.0,0.1,NaN,120.0
3,gradient_boosting,3,504091,50347,115.29,NaN,NaN,0.831172,0.566689,0.277955,0.372971,0.821238,100.0,2.0,NaN,0.1,0.8,NaN


### 3.2 Review the Hyperparameter Tuning Log

The tuning log keeps every tested hyperparameter combination together with its mean validation performance across the temporal folds. This makes the tuning process transparent and gives a concrete basis for the report discussion.


In [11]:
tuning_results = pd.read_csv(REPORTS_DIR / 'hyperparameter_tuning_results.csv')
tuning_results.head(12)


,model,num_folds,fit_seconds,C,class_weight,valid_mean_accuracy,valid_mean_precision,valid_mean_recall,valid_mean_f1,valid_mean_roc_auc,n_estimators,max_depth,min_samples_leaf,learning_rate,subsample,max_iter
0,gradient_boosting,3,219.13,NaN,NaN,0.828281,0.545148,0.296762,0.381867,0.816835,100.0,2.0,NaN,0.10,0.8,NaN
1,gradient_boosting,3,218.27,NaN,NaN,0.828440,0.558749,0.242577,0.334928,0.813617,100.0,2.0,NaN,0.05,0.8,NaN
2,hist_gradient_boosting,3,11.02,NaN,NaN,0.830765,0.547173,0.351620,0.427767,0.825542,NaN,8.0,50.0,0.10,NaN,120.0
3,hist_gradient_boosting,3,11.08,NaN,NaN,0.830995,0.548566,0.350128,0.426963,0.825675,NaN,8.0,20.0,0.10,NaN,120.0
4,hist_gradient_boosting,3,12.64,NaN,NaN,0.831014,0.550716,0.337167,0.417120,0.824358,NaN,8.0,50.0,0.05,NaN,120.0
5,hist_gradient_boosting,3,12.89,NaN,NaN,0.830979,0.550881,0.336345,0.416540,0.824363,NaN,8.0,20.0,0.05,NaN,120.0
6,logistic_regression,3,10.45,0.1,balanced,0.723290,0.368438,0.755310,0.493814,0.813331,NaN,NaN,NaN,NaN,NaN,NaN
7,logistic_regression,3,11.55,1.0,balanced,0.722889,0.368112,0.755609,0.493539,0.813289,NaN,NaN,NaN,NaN,NaN,NaN
8,logistic_regression,3,11.22,0.1,NaN,0.824418,0.525390,0.325476,0.399864,0.812676,NaN,NaN,NaN,NaN,NaN,NaN
9,logistic_regression,3,11.65,1.0,NaN,0.824442,0.525405,0.325024,0.399618,0.812719,NaN,NaN,NaN,NaN,NaN,NaN


### 3.3 Review the Time-Fold Validation Metrics

In addition to the mean tuning summary, the notebook also saves per-fold validation results. This table reports the performance of each parameter combination on each temporal fold, including F1-score, ROC-AUC, recall, and precision.


In [12]:
fold_metrics = pd.read_csv(REPORTS_DIR / 'time_fold_metrics.csv')
fold_metrics[['model', 'fold', 'valid_start_date', 'valid_end_date', 'fold_f1', 'fold_roc_auc', 'fold_recall', 'fold_precision']].head(12)


,model,fold,valid_start_date,valid_end_date,fold_f1,fold_roc_auc,fold_recall,fold_precision
0,gradient_boosting,1,2020-01-01,2021-08-31,0.256362,0.823767,0.169193,0.528809
1,gradient_boosting,1,2020-01-01,2021-08-31,0.312431,0.829262,0.224763,0.512220
2,gradient_boosting,2,2021-09-01,2023-05-02,0.325254,0.806926,0.230860,0.550235
3,gradient_boosting,2,2021-09-01,2023-05-02,0.370284,0.809207,0.281615,0.540449
4,gradient_boosting,3,2023-05-03,2024-12-31,0.423169,0.810159,0.327678,0.597204
5,gradient_boosting,3,2023-05-03,2024-12-31,0.462886,0.812035,0.383908,0.582774
6,hist_gradient_boosting,1,2020-01-01,2021-08-31,0.353085,0.834589,0.269609,0.511432
7,hist_gradient_boosting,1,2020-01-01,2021-08-31,0.352968,0.834572,0.269786,0.510310
8,hist_gradient_boosting,1,2020-01-01,2021-08-31,0.372917,0.835852,0.296552,0.502252
9,hist_gradient_boosting,1,2020-01-01,2021-08-31,0.375993,0.835915,0.302047,0.497882


### 3.4 Inspect the Selected Best Parameters

The selected best hyperparameters for each model are stored separately so that the final training configuration is explicit and easy to reproduce later.


In [13]:
best_params = json.loads((REPORTS_DIR / 'best_params.json').read_text(encoding='utf-8'))
best_params


{'logistic_regression': {'C': 0.1, 'class_weight': 'balanced'},
 'random_forest': {'n_estimators': 100,
  'max_depth': 12,
  'min_samples_leaf': 5,
  'class_weight': 'balanced_subsample'},
 'gradient_boosting': {'n_estimators': 100,
  'learning_rate': 0.1,
  'max_depth': 2,
  'subsample': 0.8},
 'hist_gradient_boosting': {'max_iter': 120,
  'learning_rate': 0.1,
  'max_depth': 8,
  'min_samples_leaf': 50}}

### 3.5 Inspect a Sample Prediction File

Each prediction file contains the district, date, time block, true label, predicted probability, and final predicted class. This format is suitable for both evaluation and downstream error analysis.


In [14]:
sample_prediction = pd.read_csv(PREDICTIONS_DIR / 'logistic_regression_test_predictions.csv')
sample_prediction.head()


,district,event_date,time_block,actual_label,predicted_probability,predicted_label
0,1,2025-01-01,0,0,0.058135,0
1,1,2025-01-01,1,0,0.416987,0
2,1,2025-01-01,2,1,0.544872,1
3,1,2025-01-01,3,0,0.775609,1
4,1,2025-01-01,4,0,0.542595,1


## 4. Report Writing Support

### Why this section is needed
To keep the final report consistent with the notebook outputs, this section generates ready-to-use report text for the required parts: `Model Implementation`, `Training Process`, and `Hyperparameter Tuning`. The text is based on the actual workflow and saved outputs from this notebook.

### Output
- a structured markdown file containing report-ready paragraphs
- a notebook preview of the generated report text


In [15]:
best_model_row = metrics.sort_values(['test_f1', 'test_roc_auc'], ascending=[False, False]).iloc[0]
best_model_name = best_model_row['model']
feature_count = train_x.shape[1]

report_text = f"""# Model Implementation

Four classification models were implemented for hotspot prediction: Logistic Regression, Random Forest, Gradient Boosting, and HistGradientBoosting. These models were selected to provide a balanced comparison between an interpretable linear baseline and more flexible ensemble-based classifiers. Logistic Regression was included as a simple and transparent reference model, while the tree-based ensemble models were included because they can capture non-linear relationships among spatial, temporal, and historical crime features. The modeling target was `target_hotspot_next_block`, which indicates whether a district becomes a hotspot in the next 4-hour block.

The model input was based on the district-level featured panel prepared in Phase 2. After excluding the target column, dataset marker, and raw date field from direct model input, the feature matrix contained {feature_count} encoded predictors. These predictors included district identity, time block, day of week, month, quarter, weekend indicator, current crime count, current hotspot status, lagged counts from previous blocks, and rolling historical averages. Categorical fields were converted using one-hot encoding so that all four models could be trained on a consistent feature representation.

# Training Process

The training workflow used `train_panel_featured.csv` as the development dataset and `test_panel_featured.csv` as the final evaluation dataset. To preserve the chronological structure of the crime forecasting task, a multi-fold temporal validation strategy was used instead of a random train-validation split. Specifically, the training period was divided into three expanding time folds. In each fold, earlier observations were used to train the model and the immediately following time block was used as the validation subset. This design reduces the risk of information leakage from future periods into earlier observations and better matches the intended real-world prediction setting.

After selecting the best hyperparameters for each model, the chosen configuration was retrained on the full 2015-2024 training dataset and then applied to the 2025 test set. For each model, the notebook saved a serialized model artifact in the `models/` folder and a corresponding prediction file in the `predictions/` folder. Final model performance was compared using accuracy, precision, recall, F1-score, and ROC-AUC. Among the evaluated models, `{best_model_name}` achieved the strongest overall test performance under the current selection rule based on F1-score.

# Hyperparameter Tuning

Hyperparameter tuning was carried out through a compact grid search for each of the four models. For Logistic Regression, the search focused on regularization strength and class weighting. For Random Forest, the tuning considered tree depth and minimum leaf size under a balanced-subsample setting. For Gradient Boosting, the search varied the learning rate while keeping a stable number of boosting stages and shallow trees. For HistGradientBoosting, the tuning considered the learning rate and minimum leaf size under a fixed iteration budget and tree depth.

Each hyperparameter combination was evaluated across all temporal folds, and for each combination the fold-level F1-score, ROC-AUC, recall, and precision were recorded separately. The parameter set with the highest mean validation F1-score across folds was selected as the final configuration for that model. The fold-level results were exported to `reports/time_fold_metrics.csv`, the mean tuning summary was exported to `reports/hyperparameter_tuning_results.csv`, and the selected parameter sets were exported to `reports/best_params.json`. This makes the tuning process transparent, reproducible, and directly traceable to the saved notebook outputs."""

report_path = REPORTS_DIR / 'model_training_report_sections.md'
report_path.write_text(report_text, encoding='utf-8')
print(f'Saved report section draft to: {report_path}')
print()
print(report_text)


Saved report section draft to: /home/autolife/Documents/rossi_new/reports/model_training_report_sections.md

# Model Implementation

Four classification models were implemented for hotspot prediction: Logistic Regression, Random Forest, Gradient Boosting, and HistGradientBoosting. These models were selected to provide a balanced comparison between an interpretable linear baseline and more flexible ensemble-based classifiers. Logistic Regression was included as a simple and transparent reference model, while the tree-based ensemble models were included because they can capture non-linear relationships among spatial, temporal, and historical crime features. The modeling target was `target_hotspot_next_block`, which indicates whether a district becomes a hotspot in the next 4-hour block.

The model input was based on the district-level featured panel prepared in Phase 2. After excluding the target column, dataset marker, and raw date field from direct model input, the feature matrix conta